# SFT 実習：一次方程式の解答を学習する

英語の一次方程式 Q&A で、Qwen2.5-1.5B-Instruct を教師ありファインチューニング（SFT）します。「答えのみ」と「途中式付き」で別々の QLoRA アダプターを学習し、同じ問題で比較します。結果を読むセルは、保存済みのデータと実測記録を使うため GPU がなくても実行できます。

学習には **Unsloth（アンスロス）** を使います。

- [Q&A データセットと学習・評価](8300.0180.LLM_QA_datasets.ipynb)
- [使用した4ビットモデル](https://huggingface.co/unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit)

## 1. 比較に使う Q&A

学習1,000問、検証100問、テスト200問を使います。各分割は、片辺にだけ変数がある式と両辺に変数がある式を半数ずつ含みます。学習・検証では、2条件で質問とその順番をそろえ、模範回答のみを変えます。[データ生成コード](examples/SFT/generate_data.py) では同じ式が別の分割に入らないよう正規化して比較しています。

先に整数解を選び、$ax+b=d$ または $ax+b=cx+d$ を組み立てます。$a=c$ は除き、係数や解に負の値も含めます。学習用の模範回答は2条件で計2,000件です。質問を `user`、模範回答を `assistant` とする `messages` 形式で保存します。

次のセルで、データに収録された同じ質問に対する2種類の模範回答を表示します。

In [ ]:
import json
from pathlib import Path

example_dir = Path("examples/SFT")
run_dir = example_dir / "runs" / "31820"


def read_jsonl(path):
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream]


for condition in ("answer_only", "worked"):
    examples = read_jsonl(example_dir / "data" / f"train_{condition}.jsonl")
    print(f"[{condition}]")
    for message in examples[0]["messages"]:
        print(f"{message['role']}: {message['content']}")
    print()

## 2. 学習と採点の設定

4ビットで読み込んだモデルの重みを固定し、LoRA アダプターを学習します。各条件は同じ初期モデルから始め、問題の順番と学習設定をそろえ、アダプターを別々に保存します。実測では、シード42、3エポック、GPU あたりバッチサイズ1、勾配累積8、学習率 `2e-4` を使用しました。質問側を損失計算から除き、模範回答側を学習します。両条件で同じチャットテンプレートを使用し、トークン化の段階で模範回答が切り捨てられていないことを確認します。[Unsloth のデータ形式](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/datasets-guide) と [TRL の SFTTrainer](https://huggingface.co/docs/trl/en/sft_trainer) を使用します。検証損失で選んだチェックポイントは両条件とも375ステップでした。 [学習コード](examples/SFT/experiment.py)

評価では、学習前と学習後の3条件で同じテスト200問を使います。貪欲デコードで最大192トークンを生成し、最後の行を `Answer: x = <integer>` として抽出します。正しい整数が得られた件数を全200問で割り、整数が違う場合と書式のため抽出できない場合は別に数えます。途中式については、各変形が元の方程式と同じ解を持つかも検査します。 [採点コード](examples/SFT/score.py)

## 3. RTX 3060 での実測

Slurm ジョブ `31820` は RTX 3060（12 GB）1基で実行し、2条件の学習と3条件の評価を含め **16分46秒**で終了しました。実行環境は Python 3.11、Unsloth 2026.11.10、PyTorch 2.14.1（CUDA 13.0）、Transformers 5.17.0、TRL 1.15.0 です。 [実行時のパッケージ一覧](examples/SFT/runs/31820/packages.txt)

In [ ]:
from IPython.display import Markdown, display

conditions = {
    "baseline": "学習前",
    "answer_only": "答えのみ",
    "worked": "途中式付き",
}
evaluations = {}
rows = [
    "| 条件 | 指定形式での正答率 | 正答 | 値の誤り | 抽出失敗 | 片辺に変数 | 両辺に変数 | 学習時間 | 学習トークン数 |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for condition, label in conditions.items():
    result_dir = run_dir / condition
    metrics = json.loads((result_dir / "evaluation.json").read_text())["metrics"]
    evaluations[condition] = metrics
    overall = metrics["overall"]
    duration, token_count = "—", "—"
    if condition != "baseline":
        training = json.loads((result_dir / "training.json").read_text())
        duration = f"{training['wall_seconds_including_validation_and_checkpoints']:.2f} 秒"
        token_count = f"{training['training_tokens']:,}"
    rows.append(
        f"| {label} | {overall['accuracy']:.1%} | {overall['correct']} | "
        f"{overall['incorrect']} | {overall['unparseable']} | "
        f"{metrics['one_side']['correct']}/100 | {metrics['both_sides']['correct']}/100 | "
        f"{duration} | {token_count} |"
    )
display(Markdown("\n".join(rows)))

この実行では、答えのみが **61.5%（123/200）**、途中式付きが **99.0%（198/200）** でした。両条件とも全200問で最終行を抽出できました。途中式付きでは198件ですべての式変形が同値で、2件に非同値の変形がありました。

学習前は指定形式で **3.5%（7/200）** でした。193件は最終行を抽出できず、この採点からはそれらの問題を解けたか判断できません。学習時間は検証とチェックポイント保存を含みます。この1回の比較では、途中式付きの条件は処理トークン数も増えています。

次のセルは、同じテスト問題に対する学習後の実出力を表示します。

In [ ]:
equation = "x + 25 = 11"
for condition in ("answer_only", "worked"):
    predictions = read_jsonl(run_dir / condition / "predictions.jsonl")
    prediction = next(row for row in predictions if row["equation"] == equation)
    print(f"[{conditions[condition]}] {equation}")
    print(prediction["response"])
    print()

### 実行前の時間試算

デスクトップ版 RTX 3060（VRAM 12 GB）、4ビット QLoRA、学習1,000問、3エポックを想定する。チャットテンプレート込みで1問平均45／90トークン、実効学習速度100〜300トークン/秒と仮定した試算であり、この構成での実測値ではない。

| 条件 | 3エポックの処理量 | 学習時間の概算 |
|---|---:|---:|
| 答えだけ | 約13.5万トークン | 約8〜23分 |
| 途中式つき | 約27万トークン | 約15〜45分 |

2条件の学習は丸めて約25〜70分、初回のモデル取得・初期化と評価を含む作業枠は約1〜2時間を目安とする。GPU 付き PC では各条件のトークン数を計測し、ウォームアップ後の20〜30学習ステップから実効速度を測って見積もりを更新する。 [Unsloth の動作要件](https://unsloth.ai/docs/get-started/fine-tuning-for-beginners/unsloth-requirements) · [ベンチマークとウォームアップ](https://unsloth.ai/docs/basics/unsloth-benchmarks)

今回の実測値は上の表に示したとおりです。

## 4. GPU 付き PC で再実行する

以下は実測と同じ Linux・Slurm 環境での手順です。リポジトリのルートから `8300.LLM/examples/SFT` に移動して実行します。モデルのダウンロード先は計算ノードからも参照できる必要があります。

```bash
cd 8300.LLM/examples/SFT
conda create -n SFT python=3.11 pip
conda activate SFT
python -m pip install -e .
python -m pip check
python download_model.py
python generate_data.py
sbatch run_sft.sbatch
```

Slurm を使わない GPU 付き PC では、同じディレクトリと環境で直接実行できます。新しい実行ディレクトリを指定すると、今回の実測記録と並べて保存できます。

```bash
run_dir="runs/local-$(date +%Y%m%d-%H%M%S)"
mkdir -p "$run_dir"
python experiment.py train answer_only "$run_dir"
python experiment.py train worked "$run_dir"
for condition in baseline answer_only worked; do
    python experiment.py evaluate "$condition" "$run_dir"
done
```

- [依存パッケージの指定](examples/SFT/pyproject.toml)
- [Slurm ジョブスクリプト](examples/SFT/run_sft.sbatch)
- [学習前の評価 JSON](examples/SFT/runs/31820/baseline/evaluation.json)
- 答えのみの [学習記録](examples/SFT/runs/31820/answer_only/training.json) と [評価 JSON](examples/SFT/runs/31820/answer_only/evaluation.json)
- 途中式付きの [学習記録](examples/SFT/runs/31820/worked/training.json) と [評価 JSON](examples/SFT/runs/31820/worked/evaluation.json)